Wczytanie danych

In [11]:
import os, json, time, numpy as np, pandas as pd, dash
import plotly.express as px, plotly.graph_objs as go, dash_bootstrap_components as dbc
from dash import Input, Output, State, dcc, html, dash_table, ALL, MATCH, ctx
#region Start
app = dash.Dash(
    __name__,
    external_stylesheets=[
        dbc.themes.BOOTSTRAP, #Wygląd
        "https://cdn.jsdelivr.net/npm/bootstrap-icons@1.11.0/font/bootstrap-icons.css"
    ],
    suppress_callback_exceptions=True
)
DATA_FOLDER = 'Dane Dash/' #Folder do zczytania gotowych danych
frames = []  
for filename in os.listdir(DATA_FOLDER):
    if filename.endswith('.json'):
        filepath = os.path.join(DATA_FOLDER, filename)
        frames.append(pd.read_json(filepath))
df = pd.concat(frames, ignore_index=True)

Kolumny z dniem tygodnia i godziną

In [12]:
# Zmiana na czas polski
timestamp = df['timestamp']
timestamp_pl = timestamp.dt.tz_convert('Europe/Warsaw')
df['timestamp_pl'] = timestamp_pl
df['day_of_year'] = timestamp_pl.dt.day_of_year
df['weekday'] = timestamp_pl.dt.day_name('Pl')
df['weekday_number'] = timestamp_pl.dt.weekday + 1
df['hour'] = timestamp_pl.dt.hour
df.sample(2)

,timestamp,duration,app,status,device,osoba,category,timestamp_pl,day_of_year,weekday,weekday_number,hour
25610,2026-01-02 13:30:32.316000+00:00,186.691,ms-teams.exe,afk,PC,AK,Komunikatory,2026-01-02 14:30:32.316000+01:00,2,Piątek,5,14
69146,2025-12-29 09:41:30.685000+00:00,29.318,MATLAB.exe,not-afk,PC,Kamil,studia,2025-12-29 10:41:30.685000+01:00,363,Poniedziałek,1,10


In [13]:
Popular_apps = df[df['osoba']=='AK'].groupby('app')['duration'].aggregate(total_duration = 'sum').reset_index().sort_values('total_duration',ascending=False).reset_index(drop=True)
Popular_apps.head(5)

,app,total_duration
0,chrome.exe,366976.186
1,Code.exe,46365.289
2,MATLAB.exe,32554.079
3,aces.exe,23963.472
4,explorer.exe,14756.341


# Średnia aktywność w dniach tygodnia

In [22]:
# po czym agregować kategoria / aplikacja
agregator = "category"
#agregator = "app"

In [23]:
Dni = df[['osoba',"weekday","weekday_number","day_of_year"]].drop_duplicates()
Dni = Dni.groupby(["osoba","weekday","weekday_number"])['day_of_year'].aggregate(n = 'count').reset_index().sort_values(['osoba','weekday_number'])
Dni

,osoba,weekday,weekday_number,n
3,AK,Poniedziałek,1,4
5,AK,Wtorek,2,3
6,AK,Środa,3,3
0,AK,Czwartek,4,3
2,AK,Piątek,5,3
4,AK,Sobota,6,4
1,AK,Niedziela,7,3
10,J,Poniedziałek,1,5
12,J,Wtorek,2,5
13,J,Środa,3,6


In [24]:
df2 = df.groupby(["osoba","weekday","weekday_number",agregator])['duration'].aggregate(total_activity = 'sum').reset_index()
df2['total_activity'] /= 3600
# Teraz podzilić przez liczbę zarejestrowanych dni
df2 = pd.merge(df2,Dni)
df2['average_activity'] = df2['total_activity'] / df2['n']
df2

,osoba,weekday,weekday_number,category,total_activity,n,average_activity
0,AK,Czwartek,4,Komunikatory,0.645079,3,0.215026
1,AK,Czwartek,4,gry komputerowe,0.731712,3,0.243904
2,AK,Czwartek,4,inne,2.181041,3,0.727014
3,AK,Czwartek,4,nauka,0.025287,3,0.008429
4,AK,Czwartek,4,oglądanie filmów,7.625299,3,2.541766
...,...,...,...,...,...,...,...
129,Kamil,Środa,3,gry komputerowe,4.879762,4,1.219940
130,Kamil,Środa,3,inne,3.570650,4,0.892663
131,Kamil,Środa,3,nauka,0.000563,4,0.000141
132,Kamil,Środa,3,oglądanie filmów,1.399281,4,0.349820


In [25]:
# To będzie do czegoś innego
Suma_z_dnia = df.groupby(['osoba','day_of_year'])['duration'].sum().reset_index().sort_values('duration',ascending=False)
Suma_z_dnia['duration'] /= 3600
Suma_z_dnia

,osoba,day_of_year,duration
52,J,358,18.773711
24,J,2,18.524533
54,J,360,17.655674
55,J,361,17.523074
43,J,349,16.832582
...,...,...,...
38,J,344,2.476527
34,J,340,2.030039
5,AK,6,2.020986
14,AK,352,1.984784


In [108]:
df3.head(1)

,osoba,weekday,weekday_number,category,total_activity,n,average_activity
0,AK,Czwartek,4,Komunikatory,0.645079,3,0.215026


In [110]:
df3['average_activity_hm'] = df3['average_activity'].apply(lambda x: f"{int(x)}h {int(((x%1)*60))}m")

C:\Users\Antek\AppData\Local\Temp\ipykernel_17936\2140339675.py:1: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [ ]:
osoba = "AK"
df3 = df2[df2['osoba'] == osoba]
ylim = np.ceil(df2.groupby(['osoba','weekday'])['average_activity'].sum().reset_index()['average_activity'].max())
df3['average_activity_hm'] = df3['average_activity'].apply(lambda x: f"{int(x)}h {int(((x%1)*60))}m")
fig = px.bar(df3,
       x='weekday',
       y = 'average_activity',
       color=agregator,
       category_orders={'weekday':['Poniedziałek','Wtorek','Środa','Czwartek','Piątek','Sobota','Niedziela']},
       labels={'weekday_number':'dzień tygodnia','average_activity':'średnia aktywność (godz.)',agregator:agregator},
       title=f"Średnia aktywność w dniach tygodnia: {osoba}",
       template='plotly_white',
       hover_data={
       agregator: True,                  # pokaż nazwę aplikacji/kategorii
       'average_activity_hm': True,      # pokaż czas w Hh Mm
       'average_activity': True,          # ukryj dziesiętne godziny
       'weekday':False
       },
       range_y=[0,ylim])

for trace in fig.data:
    # bierzemy tylko wiersze df3, które pasują do tego trace (trace.name = nazwa grupy)
    subset = df3[df3[agregator] == trace.name].copy()
    trace.customdata = np.stack(
        (subset[agregator], subset['average_activity_hm']), axis=-1
    )
    trace.hovertemplate = (
        "<b>%{customdata[0]}</b><br>"
        "Czas średni: %{customdata[1]}"
        "<extra></extra>"
    )
fig.show()

C:\Users\Antek\AppData\Local\Temp\ipykernel_17936\564463176.py:4: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [ ]:
df3

,osoba,weekday,weekday_number,category,total_activity,n,average_activity
0,AK,Czwartek,4,Komunikatory,0.645079,3,0.215026
1,AK,Czwartek,4,gry komputerowe,0.731712,3,0.243904
2,AK,Czwartek,4,inne,2.181041,3,0.727014
3,AK,Czwartek,4,nauka,0.025287,3,0.008429
4,AK,Czwartek,4,oglądanie filmów,7.625299,3,2.541766
5,AK,Czwartek,4,studia,5.572816,3,1.857605
6,AK,Niedziela,7,Komunikatory,0.217509,3,0.072503
7,AK,Niedziela,7,gry komputerowe,0.842681,3,0.280894
8,AK,Niedziela,7,inne,5.238338,3,1.746113
9,AK,Niedziela,7,nauka,0.308262,3,0.102754


In [29]:
COLOURS = {'muzyka': '#5544ee', 
           'nauka': "#df6092", 
           'studia': '#dd5544', 
           'Komunikatory' : '#77ee44',
           'gry komputerowe': '#8855aa',
           'oglądanie filmów': "#F7D83B",
           'inne': '#00c0ff'}
from hashlib import md5
from colorsys import hsv_to_rgb
def get_colour(text: str) -> str:
    if text in COLOURS.keys():
        return COLOURS[text]
    else:
        h = int(md5(text.encode("utf-8")).hexdigest(), 16)
        return "rgb(%d,%d,%d)" % tuple([int(i*255) for i in hsv_to_rgb(((h >> 10)%360)/360, 0.7+(0.3*(((h >> 20)%360)/360)), 0.8+(0.2*(((h >> 30)%360)/360)))])


In [33]:
{i: get_colour(i) for i in df3[agregator]}

{'Komunikatory': '#77ee44',
 'gry komputerowe': '#8855aa',
 'inne': '#00c0ff',
 'nauka': '#df6092',
 'oglądanie filmów': '#F7D83B',
 'studia': '#dd5544'}

In [34]:
DATA_FOLDER = 'Dane Dash/' #Folder do zczytania gotowych danych
frames = []  
for filename in os.listdir(DATA_FOLDER):
    if filename.endswith('.json'):
        filepath = os.path.join(DATA_FOLDER, filename)
        frames.append(pd.read_json(filepath))
df = pd.concat(frames, ignore_index=True)

In [44]:
dates = pd.to_datetime(df['timestamp']).dt.tz_convert('Europe/Warsaw')
dates

0       2026-01-07 21:12:54.744000+01:00
1       2026-01-07 21:12:33.030000+01:00
2       2026-01-07 21:12:25.736000+01:00
3       2026-01-07 21:11:57.246000+01:00
4       2026-01-07 21:11:04.243000+01:00
                      ...               
79165   2026-01-10 22:01:53.298000+01:00
79166   2026-01-10 22:01:55.318000+01:00
79167   2026-01-10 22:02:18.447000+01:00
79168   2026-01-10 22:02:27.751000+01:00
79169   2026-01-10 22:02:32.836000+01:00
Name: timestamp, Length: 79170, dtype: datetime64[ns, Europe/Warsaw]

In [55]:
dates.dt.tz_localize(None)

0       2026-01-07 21:12:54.744
1       2026-01-07 21:12:33.030
2       2026-01-07 21:12:25.736
3       2026-01-07 21:11:57.246
4       2026-01-07 21:11:04.243
                  ...          
79165   2026-01-10 22:01:53.298
79166   2026-01-10 22:01:55.318
79167   2026-01-10 22:02:18.447
79168   2026-01-10 22:02:27.751
79169   2026-01-10 22:02:32.836
Name: timestamp, Length: 79170, dtype: datetime64[ns]

In [60]:
t = pd.Timestamp('2026-01-05')
t = t.tz_localize('Europe/Warsaw')
t

Timestamp('2026-01-05 00:00:00+0100', tz='Europe/Warsaw')

In [61]:
dates > t

0        True
1        True
2        True
3        True
4        True
         ... 
79165    True
79166    True
79167    True
79168    True
79169    True
Name: timestamp, Length: 79170, dtype: bool

In [62]:
dates

0       2026-01-07 21:12:54.744000+01:00
1       2026-01-07 21:12:33.030000+01:00
2       2026-01-07 21:12:25.736000+01:00
3       2026-01-07 21:11:57.246000+01:00
4       2026-01-07 21:11:04.243000+01:00
                      ...               
79165   2026-01-10 22:01:53.298000+01:00
79166   2026-01-10 22:01:55.318000+01:00
79167   2026-01-10 22:02:18.447000+01:00
79168   2026-01-10 22:02:27.751000+01:00
79169   2026-01-10 22:02:32.836000+01:00
Name: timestamp, Length: 79170, dtype: datetime64[ns, Europe/Warsaw]

In [65]:
dates.tz_localize('Europe/Warsaw')

TypeError: index is not a valid DatetimeIndex or PeriodIndex

In [72]:
dates_pl = pd.to_datetime(df['timestamp'],utc=True).dt.tz_convert('Europe/Warsaw')
dates_pl

0       2026-01-07 21:12:54.744000+01:00
1       2026-01-07 21:12:33.030000+01:00
2       2026-01-07 21:12:25.736000+01:00
3       2026-01-07 21:11:57.246000+01:00
4       2026-01-07 21:11:04.243000+01:00
                      ...               
79165   2026-01-10 22:01:53.298000+01:00
79166   2026-01-10 22:01:55.318000+01:00
79167   2026-01-10 22:02:18.447000+01:00
79168   2026-01-10 22:02:27.751000+01:00
79169   2026-01-10 22:02:32.836000+01:00
Name: timestamp, Length: 79170, dtype: datetime64[ns, Europe/Warsaw]

In [73]:
dates_pl.dt.tz_localize(None)

0       2026-01-07 21:12:54.744
1       2026-01-07 21:12:33.030
2       2026-01-07 21:12:25.736
3       2026-01-07 21:11:57.246
4       2026-01-07 21:11:04.243
                  ...          
79165   2026-01-10 22:01:53.298
79166   2026-01-10 22:01:55.318
79167   2026-01-10 22:02:18.447
79168   2026-01-10 22:02:27.751
79169   2026-01-10 22:02:32.836
Name: timestamp, Length: 79170, dtype: datetime64[ns]

In [76]:
df['timestamp'].dt.strftime('%H:%M')

0        20:12
1        20:12
2        20:12
3        20:11
4        20:11
         ...  
79165    21:01
79166    21:01
79167    21:02
79168    21:02
79169    21:02
Name: timestamp, Length: 79170, dtype: object

In [100]:
df3

,osoba,weekday,weekday_number,category,total_activity,n,average_activity
0,AK,Czwartek,4,Komunikatory,0.645079,3,0.215026
1,AK,Czwartek,4,gry komputerowe,0.731712,3,0.243904
2,AK,Czwartek,4,inne,2.181041,3,0.727014
3,AK,Czwartek,4,nauka,0.025287,3,0.008429
4,AK,Czwartek,4,oglądanie filmów,7.625299,3,2.541766
5,AK,Czwartek,4,studia,5.572816,3,1.857605
6,AK,Niedziela,7,Komunikatory,0.217509,3,0.072503
7,AK,Niedziela,7,gry komputerowe,0.842681,3,0.280894
8,AK,Niedziela,7,inne,5.238338,3,1.746113
9,AK,Niedziela,7,nauka,0.308262,3,0.102754


In [106]:
df3['average_activity'].apply(lambda x: f"{int(x)}h {int(((x%1)*60))}m")

0     0h 12m
1     0h 14m
2     0h 43m
3      0h 0m
4     2h 32m
5     1h 51m
6      0h 4m
7     0h 16m
8     1h 44m
9      0h 6m
10    3h 17m
11    3h 17m
12    0h 48m
13    0h 14m
14    0h 15m
15    3h 10m
16     1h 5m
17     0h 7m
18     0h 7m
19     1h 2m
20     0h 0m
21    2h 14m
22     2h 8m
23     0h 6m
24     0h 4m
25    1h 20m
26    1h 44m
27    3h 24m
28    0h 15m
29    0h 36m
30    1h 54m
31     0h 2m
32    2h 46m
33    0h 44m
34    0h 10m
35    0h 14m
36    1h 42m
37     0h 0m
38    2h 22m
39    1h 37m
Name: average_activity, dtype: object

In [105]:
2.5 % 1

0.5